In [10]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Step 1: Load all match summary files and merge
match_files = [
    'matches_updated_ipl_upto_2025.csv',
    'matches_updated_mens_ipl.csv'
]

matches_list = []
for file in match_files:
    df = pd.read_csv(file)
    print(f"Loaded {file}: shape {df.shape}")
    matches_list.append(df)

# Concat and drop duplicates based on matchId
all_matches = pd.concat(matches_list, ignore_index=True)
all_matches = all_matches.drop_duplicates(subset=['matchId'], keep='last')  # Keep most recent if dupes

# Clean columns: standardize names, fill NaNs, convert dtypes
all_matches.columns = all_matches.columns.str.lower().str.strip()
all_matches['date'] = pd.to_datetime(all_matches['date'], errors='coerce')
all_matches['season'] = all_matches['season'].str.replace('/0', '').astype(str)  # Clean seasons like 2007/08 to 2008
all_matches['winner_runs'] = pd.to_numeric(all_matches['winner_runs'], errors='coerce').fillna(0)
all_matches['winner_wickets'] = pd.to_numeric(all_matches['winner_wickets'], errors='coerce').fillna(0)
all_matches['matchid'] = all_matches['matchid'].astype(int, errors='ignore')

# Drop redundant/unused columns if exist
drop_cols = [col for col in ['date1', 'date2', 'neutralvenue', 'eliminator', 'gender', 'outcome', 'balls_per_over', 'method'] if col in all_matches.columns]
all_matches = all_matches.drop(columns=drop_cols, errors='ignore')

# Fill NaNs logically
all_matches['city'] = all_matches['city'].fillna(all_matches['venue'].str.split(',').str[0])
all_matches['winner'] = all_matches['winner'].fillna('No Result')

print("Cleaned matches shape:", all_matches.shape)
print("Sample:", all_matches.head(2))

# Save cleaned matches
all_matches.to_csv('clean_matches.csv', index=False)

Loaded matches_updated_ipl_upto_2025.csv: shape (1169, 28)
Loaded matches_updated_mens_ipl.csv: shape (1024, 28)
Cleaned matches shape: (1169, 20)
Sample:      season                                     venue                  event  \
1024   2024  MA Chidambaram Stadium, Chepauk, Chennai  Indian Premier League   
1025   2024                     Eden Gardens, Kolkata  Indian Premier League   

      winner_runs    umpire2                  toss_winner       date  \
1024          0.0  VK Sharma  Royal Challengers Bengaluru 2024-03-22   
1025          4.0   YC Barde          Sunrisers Hyderabad 2024-03-23   

         umpire1     city reserve_umpire                 winner  \
1024  HAS Khalid  Chennai     M Kuppuraj    Chennai Super Kings   
1025    R Pandit  Kolkata   M Krishnadas  Kolkata Knight Riders   

                            team1 toss_decision                team2  \
1024  Royal Challengers Bengaluru           bat  Chennai Super Kings   
1025        Kolkata Knight Riders        

In [11]:
# Step 2: Load all deliveries (ball-by-ball) files and merge
deliveries_files = [
    'deliveries_updated_mens_ipl.csv',
    'deliveries_updated_ipl_upto_2025.csv',
    'IPL_ball_by_ball_updated.csv'
]

deliveries_list = []
for file in deliveries_files:
    df = pd.read_csv(file)
    print(f"Loaded {file}: shape {df.shape}")
    deliveries_list.append(df)

# Concat and drop duplicates based on matchId + inning + over + ball
all_deliveries = pd.concat(deliveries_list, ignore_index=True)
all_deliveries = all_deliveries.drop_duplicates(subset=['matchId', 'inning', 'over', 'ball'], keep='last')

# Clean columns: standardize, fix dtypes
all_deliveries.columns = all_deliveries.columns.str.lower().str.strip()
all_deliveries['matchid'] = all_deliveries['matchid'].astype(int, errors='ignore')
all_deliveries['date'] = pd.to_datetime(all_deliveries['date'], errors='coerce')
all_deliveries['iswide'] = pd.to_numeric(all_deliveries['iswide'], errors='coerce').fillna(0)
all_deliveries['isnoball'] = pd.to_numeric(all_deliveries['isnoball'], errors='coerce').fillna(0)
all_deliveries['batsman_runs'] = pd.to_numeric(all_deliveries['batsman_runs'], errors='coerce').fillna(0)
all_deliveries['extras'] = pd.to_numeric(all_deliveries['extras'], errors='coerce').fillna(0)
all_deliveries['total_runs'] = all_deliveries['batsman_runs'] + all_deliveries['extras']  # Compute if missing

# Handle dismissal: standardize 'Not Out' or empty to 'Not Out'
all_deliveries['player_dismissed'] = all_deliveries['player_dismissed'].fillna('Not Out')
all_deliveries['dismissal_kind'] = all_deliveries['dismissal_kind'].fillna('Not Out')

# Add 'is_valid_ball' for aggregation later (exclude wides/noballs for balls faced)
all_deliveries['is_valid_ball'] = ((all_deliveries['iswide'] == 0) & (all_deliveries['isnoball'] == 0)).astype(int)

print("Cleaned deliveries shape:", all_deliveries.shape)
print("Sample:", all_deliveries.head(2))

# Save cleaned deliveries
all_deliveries.to_csv('clean_deliveries.csv', index=False)

Loaded deliveries_updated_mens_ipl.csv: shape (243817, 20)
Loaded deliveries_updated_ipl_upto_2025.csv: shape (278205, 20)
Loaded IPL_ball_by_ball_updated.csv: shape (243815, 22)
Cleaned deliveries shape: (278352, 37)
Sample:          matchid  inning  over_ball  over  ball           batting_team  \
243817  335982.0     1.0        0.1   0.0   1.0  Kolkata Knight Riders   
243818  335982.0     1.0        0.2   0.0   2.0  Kolkata Knight Riders   

                       bowling_team      batsman  non_striker   bowler  ...  \
243817  Royal Challengers Bangalore   SC Ganguly  BB McCullum  P Kumar  ...   
243818  Royal Challengers Bangalore  BB McCullum   SC Ganguly  P Kumar  ...   

        wides  noballs  byes  legbyes  penalty  wicket_type  \
243817    NaN      NaN   NaN      NaN      NaN          NaN   
243818    NaN      NaN   NaN      NaN      NaN          NaN   

        other_wicket_type other_player_dismissed total_runs is_valid_ball  
243817                NaN                    Na